# Qwen3-1.7B answers and all-layer embeddings (Kaggle TPU)

This notebook uses Kaggle's TPU runtime with PyTorch/XLA. It launches one worker
per TPU runtime process; each worker loads Qwen3-1.7B and handles a
disjoint slice of the selected rows.

It reads `unified_math_dataset.jsonl`, asks the model to solve each eligible
problem, and writes answers plus all-layer prompt representations. The output
contains mean-pooled prompt embeddings and the final prompt-token embedding for
every hidden-state layer (including the input embedding output), stored in
independent shards.

The Qwen3 chat template is set to non-thinking mode so the model returns a direct
answer without a separate `<think>` block. Start with the 100-record test run,
inspect the output, then set `RUN_MODE` to `full` to process all rows whose
rendered prompts are at most 512 tokens.


In [ ]:
# Kaggle's TPU runtime should provide a torch/torch_xla-compatible stack.
%pip install -q "transformers>=4.51.0" accelerate sentencepiece


In [ ]:
import json
import os
import tempfile
from pathlib import Path
from collections import Counter

# Set before torch_xla initializes; Kaggle TPU runtime should expose PJRT_DEVICE=TPU already.
os.environ.setdefault("PJRT_DEVICE", "TPU")

import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-1.7B"
BATCH_SIZE_PER_TPU_CORE = 2
SHARD_SIZE = 64
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 512
RUN_MODE = "test"  # Start with 100 eligible rows; change to "full" after reviewing the smoke run.
TEST_LIMIT = 100

if not hasattr(torch_xla, "launch"):
    raise RuntimeError("This torch_xla build lacks torch_xla.launch; use Kaggle's TPU runtime image.")

local_candidates = [
    Path("/kaggle/input/datasets/ashbornmonarch/unified-math-dataset"),
    Path("unified_math_dataset.jsonl"),
]
DATA_PATH = next((path.resolve() for path in local_candidates if path.is_file()), None)
if DATA_PATH is None:
    kaggle_input = Path("/kaggle/input")
    kaggle_matches = sorted(kaggle_input.glob("**/unified_math_dataset.jsonl")) if kaggle_input.exists() else []
    if kaggle_matches:
        DATA_PATH = kaggle_matches[0].resolve()
if DATA_PATH is None:
    raise FileNotFoundError("Attach unified_math_dataset.jsonl as a Kaggle input.")
if RUN_MODE not in {"test", "full"}:
    raise ValueError('RUN_MODE must be either "test" or "full".')

if "/kaggle/input/" in DATA_PATH.as_posix():
    OUTPUT_ROOT = Path("/kaggle/working")
    SORTED_DATA_PATH = OUTPUT_ROOT / "unified_math_dataset_sorted_by_question_char_count.jsonl"
else:
    OUTPUT_ROOT = DATA_PATH.parent
    SORTED_DATA_PATH = DATA_PATH
OUTPUT_NAME = (
    "qwen3_1_7b_unified_output_tpu_test100_512"
    if RUN_MODE == "test"
    else "qwen3_1_7b_unified_output_tpu_full_512"
)
OUTPUT_DIR = OUTPUT_ROOT / OUTPUT_NAME

PROMPT_TEMPLATE = """Solve this math problem. Show brief reasoning, then finish with:
Final answer: <answer>

Problem:
{question}
"""

print(f"Input: {DATA_PATH}")
print(f"Sorted dataset: {SORTED_DATA_PATH}")
print(f"Run mode: {RUN_MODE}; output: {OUTPUT_DIR}")
print(f"TPU workers: detected when launched; per-worker batch size: {BATCH_SIZE_PER_TPU_CORE}")
print(f"Input prompt limit: {MAX_INPUT_TOKENS} tokens; longer prompts are excluded.")


In [ ]:
source_rows = []
with DATA_PATH.open("r", encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            source_rows.append(json.loads(line))

for row in source_rows:
    row["question_char_count"] = len(row["question"])
all_rows = sorted(source_rows, key=lambda row: row["question_char_count"], reverse=True)

# Count actual Qwen prompt tokens and prepare a sorted, TPU-visible JSONL.
length_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
PROMPT_TOKEN_COUNTS = {}
for start in tqdm(range(0, len(all_rows), 256), desc="Counting Qwen prompt tokens"):
    chunk = all_rows[start:start + 256]
    prompts = [
        length_tokenizer.apply_chat_template(
            [{"role": "user", "content": PROMPT_TEMPLATE.format(question=row["question"])}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for row in chunk
    ]
    encoded = length_tokenizer(prompts, add_special_tokens=True, truncation=False, padding=False)
    for row, input_ids in zip(chunk, encoded["input_ids"]):
        row["prompt_tokens"] = len(input_ids)
        PROMPT_TOKEN_COUNTS[row["example_id"]] = len(input_ids)
del length_tokenizer

SORTED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(
    mode="w", encoding="utf-8", newline="\n", delete=False,
    dir=SORTED_DATA_PATH.parent, suffix=".tmp",
) as sorted_stream:
    sorted_temp_path = Path(sorted_stream.name)
    for order_index, row in enumerate(all_rows):
        row["dataset_order_index"] = order_index
        sorted_stream.write(json.dumps(row, ensure_ascii=False) + "\n")
os.replace(sorted_temp_path, SORTED_DATA_PATH)

eligible_rows = [row for row in all_rows if row["prompt_tokens"] <= MAX_INPUT_TOKENS]
excluded_over_limit = len(all_rows) - len(eligible_rows)
selected_rows = eligible_rows[:TEST_LIMIT] if RUN_MODE == "test" else eligible_rows
selected_ids = {row["example_id"] for row in selected_rows}
print(f"Dataset records: {len(all_rows):,}")
print(f"Sorted question character count: {all_rows[0]['question_char_count']:,} down to {all_rows[-1]['question_char_count']:,}")
print(f"Prompts over {MAX_INPUT_TOKENS} tokens excluded: {excluded_over_limit:,}")
print(f"Eligible records: {len(eligible_rows):,}; selected for {RUN_MODE}: {len(selected_rows):,}")


In [ ]:
# TPU devices are created inside each spawned worker, not in the notebook's parent process.
print("PyTorch/XLA is available. Each worker will load one Qwen replica on its XLA device.")
print("Worker launch and TPU execution happen in the final code cell.")


In [ ]:
def run_tpu_worker(
    rank,
    dataset_path,
    selected_ids,
    output_dir,
    model_id,
    prompt_template,
    max_input_tokens,
    max_new_tokens,
    batch_size,
    shard_size,
):
    import json
    from collections import Counter
    from pathlib import Path

    import torch
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from tqdm.auto import tqdm

    world_size = xr.world_size()
    if rank < 0 or rank >= world_size:
        raise RuntimeError(f"Invalid worker rank {rank} for TPU world size {world_size}.")
    device = xm.xla_device()
    output_dir = Path(output_dir)

    dataset_rows = []
    with Path(dataset_path).open("r", encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                row = json.loads(line)
                if row["example_id"] in selected_ids:
                    dataset_rows.append(row)
    worker_rows = dataset_rows[rank::world_size]

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    ).to(device)
    model.eval()

    pending_rows, pending_mean, pending_last = [], [], []
    shard_summaries = []
    split_counts = Counter()
    shard_index = 0
    total_written = 0
    metadata_path = output_dir / f"metadata_worker_{rank:02d}.jsonl"

    def flush_shard(metadata_stream):
        nonlocal shard_index, total_written
        if not pending_rows:
            return
        shard_name = f"embeddings_tpu{rank:02d}_{shard_index:05d}.pt"
        payload = {
            "example_ids": [row["example_id"] for row in pending_rows],
            "layer_embeddings": torch.stack(pending_mean).contiguous(),
            "layer_last_token_embeddings": torch.stack(pending_last).contiguous(),
        }
        torch.save(payload, output_dir / shard_name)
        for row_number, row in enumerate(pending_rows):
            row["embedding_shard"] = shard_name
            row["embedding_row"] = row_number
            metadata_stream.write(json.dumps(row, ensure_ascii=False) + "\n")
        metadata_stream.flush()
        shard_summaries.append({
            "file": shard_name,
            "count": len(pending_rows),
            "layer_embedding_shape": list(payload["layer_embeddings"].shape[1:]),
            "dtype": str(payload["layer_embeddings"].dtype),
            "worker": rank,
        })
        total_written += len(pending_rows)
        pending_rows.clear()
        pending_mean.clear()
        pending_last.clear()
        shard_index += 1

    with metadata_path.open("w", encoding="utf-8", newline="\n") as metadata_stream:
        for start in tqdm(range(0, len(worker_rows), batch_size), desc=f"TPU core {rank}"):
            batch_rows = worker_rows[start:start + batch_size]
            prompts = [
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt_template.format(question=row["question"])}],
                    tokenize=False,
                    add_generation_prompt=True,
            enable_thinking=False,
                )
                for row in batch_rows
            ]
            encoded = tokenizer(
                prompts,
                padding="max_length",
                max_length=max_input_tokens,
                truncation=False,
                return_tensors="pt",
            )
            if encoded["input_ids"].shape[1] > max_input_tokens:
                raise ValueError("A prompt exceeded the 512-token cap; refusing to truncate it.")
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)

            with torch.inference_mode():
                forward = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
                if forward.hidden_states is None:
                    raise RuntimeError("Model did not return hidden states.")
                layer_stack = torch.stack(forward.hidden_states, dim=1)
                mask = attention_mask[:, None, :, None].to(layer_stack.dtype)
                mean_layers = (layer_stack * mask).sum(dim=2) / mask.sum(dim=2).clamp_min(1)
                last_layers = layer_stack[:, :, -1, :]
                del forward, layer_stack, mask
                xm.mark_step()

                generated = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )
                xm.mark_step()
                answer_ids = generated[:, input_ids.shape[1]:].cpu()
                answers = tokenizer.batch_decode(answer_ids, skip_special_tokens=True)

                mean_cpu = mean_layers.detach().cpu().to(torch.float16)
                last_cpu = last_layers.detach().cpu().to(torch.float16)
                del generated, answer_ids, input_ids, attention_mask, encoded
                del mean_layers, last_layers

            for row, answer, mean_embedding, last_embedding in zip(
                batch_rows, answers, mean_cpu, last_cpu
            ):
                result = dict(row)
                result["qwen_model"] = model_id
                result["qwen_response"] = answer.strip()
                other_responses = [
                    item for item in row.get("responses", [])
                    if item.get("model") != model_id
                ]
                result["responses"] = other_responses + [{
                    "model": model_id,
                    "text": answer.strip(),
                }]
                pending_rows.append(result)
                pending_mean.append(mean_embedding)
                pending_last.append(last_embedding)
                split_counts[row["split"]] += 1
                if len(pending_rows) >= shard_size:
                    flush_shard(metadata_stream)
            del answers, mean_cpu, last_cpu

        flush_shard(metadata_stream)

    summary = {
        "worker": rank,
        "world_size": world_size,
        "total_records": total_written,
        "split_counts": dict(split_counts),
        "shards": shard_summaries,
    }
    (output_dir / f"worker_summary_{rank:02d}.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    del model
    xm.mark_step()


In [ ]:
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output folder already contains files: {OUTPUT_DIR}. "
        "Choose a new output folder before starting this TPU run."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# torch_xla.launch starts workers for the devices available in the TPU runtime.
torch_xla.launch(
    run_tpu_worker,
    args=(
        str(SORTED_DATA_PATH),
        selected_ids,
        str(OUTPUT_DIR),
        MODEL_ID,
        PROMPT_TEMPLATE,
        MAX_INPUT_TOKENS,
        MAX_NEW_TOKENS,
        BATCH_SIZE_PER_TPU_CORE,
        SHARD_SIZE,
    ),
)

# Consolidate per-core metadata while retaining each core's independently saved embedding shards.
all_metadata = []
all_shards = []
worker_summary_paths = sorted(OUTPUT_DIR.glob("worker_summary_*.json"))
if not worker_summary_paths:
    raise FileNotFoundError("No TPU worker summary files were created.")
for worker_summary_path in worker_summary_paths:
    rank = int(worker_summary_path.stem.rsplit("_", 1)[1])
    worker_metadata_path = OUTPUT_DIR / f"metadata_worker_{rank:02d}.jsonl"
    if not worker_metadata_path.is_file() or not worker_summary_path.is_file():
        raise FileNotFoundError(f"TPU worker {rank} did not produce its expected output files.")
    with worker_metadata_path.open("r", encoding="utf-8") as stream:
        all_metadata.extend(json.loads(line) for line in stream if line.strip())
    summary = json.loads(worker_summary_path.read_text(encoding="utf-8"))
    all_shards.extend(summary["shards"])
    worker_metadata_path.unlink()
    worker_summary_path.unlink()

all_metadata.sort(key=lambda row: row["dataset_order_index"])
metadata_path = OUTPUT_DIR / "metadata.jsonl"
with metadata_path.open("w", encoding="utf-8", newline="\n") as stream:
    for row in all_metadata:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
split_counts = Counter(row["split"] for row in all_metadata)
manifest = {
    "model": MODEL_ID,
    "backend": "PyTorch/XLA",
    "run_mode": RUN_MODE,
    "sort_order": "question_char_count_desc",
    "sorted_dataset_path": str(SORTED_DATA_PATH),
    "raw_dataset_records": len(all_rows),
    "eligible_records_at_or_below_token_limit": len(eligible_rows),
    "selected_records": len(selected_rows),
    "total_records": len(all_metadata),
    "split_counts": dict(split_counts),
    "tpu_workers": len(worker_summary_paths),
    "batch_size_per_worker": BATCH_SIZE_PER_TPU_CORE,
    "effective_batch_size": len(worker_summary_paths) * BATCH_SIZE_PER_TPU_CORE,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "excluded_over_token_limit": excluded_over_limit,
    "embedding_definition": (
        "Mean-pooled non-padding prompt token representations and final prompt token "
        "representations at every hidden-state layer, including the input embedding output."
    ),
    "shards": sorted(all_shards, key=lambda item: item["file"]),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "run_mode": RUN_MODE,
    "records_written": len(all_metadata),
    "excluded_over_512_tokens": excluded_over_limit,
    "shards": len(all_shards),
}, indent=2))


## TPU run notes

- Select a Kaggle TPU accelerator, such as TPU v5e-8, in Settings before running. This notebook uses PyTorch/XLA and launches one worker per TPU runtime process.
- It sorts by `question_char_count` descending, then excludes rendered Qwen prompts over 512 tokens. Accepted inputs are padded to a static length of 512 for XLA.
- `RUN_MODE = "test"` answers and embeds the first 100 eligible records; test outputs go to `/kaggle/working/qwen3_1_7b_unified_output_tpu_test100_512`.
- After reviewing the test output, set `RUN_MODE = "full"` and run all cells again. Full outputs go to `/kaggle/working/qwen3_1_7b_unified_output_tpu_full_512`.
- Each TPU worker writes separate embedding shards; the final cell combines metadata and records the per-worker shards in `manifest.json`.
- TPU generation can compile more slowly than GPU generation, especially on early batches. The 100-record run is meant to check that the model and output format work on the selected TPU runtime before the full run.
